# M1 — LoRA fine-tuning of Qwen3-1.7B for exercise field extraction

**SI4006 · Entrega M1 — Fine-tuning baseline del proyecto**

This notebook teaches a small language model to read an exercise description and
return its catalog fields as JSON:

> `Name: cable incline pushdown` → `{"target": "lats", "equipment": "cable"}`

It runs end to end on a **free Colab T4**. Sections:

1. Setup
2. Base model and tokenizer — and why this model
3. Dataset preparation
4. Baselines (rule-based, and the same model *without* fine-tuning)
5. LoRA configuration and training
6. Final evaluation against the baselines
7. Qualitative examples and an honest reading

Everything is seeded (`seed = 42`). Cells are meant to be run in order, top to bottom.

## 1 · Setup

Clones the project (with the dataset submodule) when running on Colab, and installs
the training stack on top of whatever torch Colab already ships.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/iamcroody/models-for-exercises-dataset.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("models-for-exercises-dataset").exists():
        subprocess.run(
            ["git", "clone", "--recurse-submodules", REPO_URL], check=True
        )
    os.chdir("models-for-exercises-dataset")
    # torch comes with the Colab image; only the training stack is missing.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=5.0", "trl>=1.9", "peft>=0.20", "datasets", "scikit-learn"],
        check=True,
    )
else:
    # Local run: notebooks/ lives one level below the repo root.
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, str(Path("scripts").resolve()))
print("working directory:", Path.cwd())

In [ ]:
import torch

import exlib

device, DTYPE = exlib.pick_device_dtype()
BF16 = str(DTYPE).endswith("bfloat16")

print(f"torch    {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu      {props.name} ({props.total_memory / 1e9:.1f} GB, sm_{props.major}{props.minor})")
print(f"dtype    {DTYPE}")
print()
print(
    "Precision is picked from the device, not hard-coded. Colab's free T4 is Turing\n"
    "(sm_75) and has no bf16 units, while TRL's SFTConfig defaults bf16=True whenever\n"
    "fp16 is unset — so a hard-coded config crashes on exactly the GPU this notebook\n"
    "is required to run on. FlashAttention-2 is Ampere+ only and is left off for the\n"
    "same reason."
)

## 2 · Base model and tokenizer

**Family: decoder.** Not because it scores best in isolation — a `DeBERTa-v3`-style
encoder classifier would very likely beat it on a closed 19-class problem — but because
the assignment states this model carries into M2 (RAG on top of it) and M3 (a visual
component). An encoder classifier can host neither. Choosing the marginally weaker
architecture that survives two more modules is the cheaper decision.

**Model: `Qwen/Qwen3-1.7B`.**

- **Apache-2.0 and ungated.** No Hugging Face login, no licence click-through. Llama-3.2
  is gated, which would break the "anyone can open this notebook and run it"
  requirement outright.
- **Fits the free tier with room.** ~1.0% of parameters trainable under LoRA, ~6.7 GB
  peak VRAM against a T4's 15 GB.
- **Has a forward path.** `Qwen3-VL-2B` shares this tokenizer and chat template, so M3's
  visual component is a family swap rather than a rewrite — and the dataset already
  ships a thumbnail and an animation GIF for all 1324 exercises.

### Tokenization check (Week 3, Lab A)

Before committing to a base model, look at how it splits the vocabulary the task turns
on. That matters more than usual here, because our labels are the **output**: the model
must reproduce all 19 target and 28 equipment values verbatim, so label fragmentation is
a direct measure of how hard the output space is.

In [ ]:
!python scripts/00_tokenizer_check.py

Read honestly: **BERT's lowercase WordPiece fragments our anatomy vocabulary the least**,
and `flan-t5` fragments it worst by a wide margin. Qwen3 sits at the front of the decoder
options but does not win outright.

So this study did not pick the model — the family requirement did. What it does establish
is that Qwen3 costs us nothing *within* its family, and it rules out the encoder-decoder
option on a concrete measurement rather than on taste.

In [ ]:
from transformers import AutoTokenizer

tokenizer = exlib.load_tokenizer(exlib.BASE_MODEL)
print(f"{exlib.BASE_MODEL}: vocab {len(tokenizer)}, eos {tokenizer.eos_token!r}, pad {tokenizer.pad_token!r}")

## 3 · Dataset

1324 exercises from a pinned git submodule. Full description, licence and known biases:
[`docs/DATASET.md`](../docs/DATASET.md).

`scripts/02_prepare_data.py` builds the splits: stratified on `target`, seed 42,
written as TRL conversational prompt/completion pairs.

In [ ]:
!python scripts/02_prepare_data.py

### The model predicts two fields, not three

The catalog also has `body_part`, but it is **not** a third prediction:

- `target → body_part` is a strict function — all 19 targets map to exactly one of 10
  body parts, no exceptions in 1324 records.
- `category` is a verbatim copy of `body_part` in every record.

So `body_part` is free whenever `target` is right. Generating it would add an accuracy
column that inflates the headline number while measuring nothing, so it is derived from a
lookup table instead. `exlib.build_body_part_map` raises if a dataset bump ever makes
`target` ambiguous, so the assumption cannot rot unnoticed.

In [ ]:
meta = exlib.load_meta()
train_rows = exlib.load_split("train")
val_rows = exlib.load_split("val")

print("splits:", meta["counts"])
print("predicted:", meta["predict_fields"], "| derived:", meta["derived_field"])
print()
print("=== one training example, exactly as the model sees it ===")
print(train_rows[0]["prompt"][0]["content"])
print()
print("--- expected completion ---")
print(train_rows[0]["completion"][0]["content"])

The prompt lists every valid value on purpose. The **same** prompt grades the untrained
model in section 4, and a base model that has never seen our labelling conventions would
otherwise be marked down for vocabulary it was never shown — which measures our
conventions, not the model.

One definition of that prompt exists (`exlib.build_prompt`) and all three evaluations
call it. If training and evaluation built the string separately, the reported delta would
partly measure prompt drift.

In [ ]:
import statistics

lengths = [
    len(tokenizer.encode(
        tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=False),
        add_special_tokens=False,
    ))
    for r in train_rows
]
print(f"tokens per training example: mean {statistics.mean(lengths):.0f}, max {max(lengths)}")
print("-> max_length = 512 truncates nothing, at half the cost of the 1024 default")

## 4 · Baselines

A single number says nothing, so there are two baselines and they measure different
things.

**(a) Rule-based** — majority class, and a substring rule that looks for the label
verbatim in the exercise name. This measures *the dataset*: how much of the task is
solvable with no model at all. Both are **fitted on train and scored on val**, exactly
like the model — a baseline that has seen the evaluation set is not a baseline.

In [ ]:
!python scripts/01_baseline.py --split val

**(b) Zero-shot — the same base model with no fine-tuning.** This is the baseline the
assignment recommends, and the only one that isolates what LoRA actually contributed.
Same prompts, same split, same greedy decoding, no adapter.

*(A few minutes on a T4.)*

In [ ]:
!python scripts/03_eval_zeroshot.py --split val

This reframes the problem. The untrained model already **beats the rule-based baseline on
both fields** and emits valid JSON every single time — it clearly reads the domain. What
it does not do is respect the closed label space: roughly a third of its `target` answers
are values we never offered. It echoes the exercise name back (`"target": "dumbbell iron
cross"`) or lands one letter off a real label (`"pectors"` for `"pectorals"`).

**Constraining output to the label space — not teaching fitness — is what the fine-tune
is for.** That is the hypothesis the rest of the notebook tests.

## 5 · LoRA configuration and training

No full fine-tuning: only low-rank adapters on the frozen base weights.

| Hyperparameter | Value | Why |
|---|---|---|
| `r` | 16 | 1081 examples over a 19+28 label space is a small, narrow target. The job is constraining output to a closed vocabulary, not installing new knowledge — that needs little capacity, and higher rank mostly buys overfitting. Tested in the ablation below rather than asserted. |
| `lora_alpha` | 32 | Holds the conventional `alpha = 2r`, so the effective scale `alpha/r` stays 2.0. Without this, changing `r` would silently change update magnitude too and confound the ablation. |
| `target_modules` | all 7 linear projections | Attention **and** MLP. Mapping "cable incline pushdown" → `{lats, cable}` is lexical-semantic, and that association lives largely in the MLP blocks; attention-only adapters can reweight what the model attends to but not what it knows a term means. |
| `lora_dropout` | 0.05 | Light regularisation on a small dataset. |
| `learning_rate` | 1e-4 | TRL's documented adapter rate, ~5× a full fine-tune's, because only the freshly-initialised low-rank matrices are learning. |
| effective batch | 16 | `per_device=2 × grad_accum=8`, fixed rather than scaled to the GPU so a local run and the Colab run take identical optimisation steps. |

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

ATTENTION = ["q_proj", "k_proj", "v_proj", "o_proj"]
MLP = ["gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=ATTENTION + MLP,
)

training_args = SFTConfig(
    output_dir="outputs/notebook",
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    max_length=512,
    seed=exlib.SEED,
    data_seed=exlib.SEED,
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    model_init_kwargs={"dtype": DTYPE},
    bf16=BF16,
    fp16=not BF16,
)

# Only the two columns TRL needs — the gold labels carried alongside in the JSONL
# for evaluation would otherwise be templated into the training text.
def to_dataset(rows):
    return Dataset.from_list(
        [{"prompt": r["prompt"], "completion": r["completion"]} for r in rows]
    )

trainer = SFTTrainer(
    model=exlib.BASE_MODEL,
    args=training_args,
    train_dataset=to_dataset(train_rows),
    eval_dataset=to_dataset(val_rows),
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

TRL computes the loss on the **completion only** for prompt-completion datasets
(`completion_only_loss` defaults to `True`), so the long label-space listing in the prompt
costs nothing at training time — the model is never asked to predict it.

Now train. *(~7 min on an RTX 5060 Ti, ~25–30 min on a Colab T4.)*

In [ ]:
train_result = trainer.train()

In [ ]:
ADAPTER_DIR = "models/notebook-r16"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

evals = [h for h in trainer.state.log_history if "eval_loss" in h]
print("eval loss per epoch: " + " -> ".join(f"{h['eval_loss']:.4f}" for h in evals))
print(f"adapter saved to {ADAPTER_DIR}")

## 6 · Final evaluation, against the same validation set

Every row of the table below is produced by the same scoring function
(`exlib.score_predictions`) on the same 121 validation records, so the comparison is not
three lookalike implementations quietly disagreeing.

Metrics and why:

- **accuracy** — closed-set classification, so exact match is the natural measure.
  `target` is the headline: 19 classes, the weakest baseline, the most room.
- **macro-F1** — accuracy hides the tail, and the tail is most of the label space
  (`levator scapulae` has 2 records in the entire catalog). Macro-F1 weights a rare class
  the same as `abs`.
- **JSON valid / in-label** — reported, never repaired. These are where the untrained
  model actually loses, and hiding them would make the delta look like magic.

In [ ]:
# Free the training copy before loading the merged model for generation.
del trainer
import gc

gc.collect()
torch.cuda.empty_cache()

!python scripts/05_eval_finetuned.py --split val --adapter {ADAPTER_DIR}

## 7 · Ablation — is `r=16` on all linear layers actually the right call?

The assignment invites experimenting rather than asserting. Three configurations,
identical apart from the variable under test. `alpha = 2r` throughout, so changing `r`
does not also change the update scale.

*(Optional — skip on a slow Colab session; results are committed in `reports/`.)*

In [ ]:
RUN_ABLATION = False  # set True to reproduce; roughly triples the notebook runtime

if RUN_ABLATION:
    !python scripts/04_train_lora.py --r 8 --run-name r8-all-linear
    !python scripts/05_eval_finetuned.py --adapter models/r8-all-linear --report-name finetuned_r8
    !python scripts/04_train_lora.py --target-modules attention --run-name r16-attention
    !python scripts/05_eval_finetuned.py --adapter models/r16-attention --report-name finetuned_attn
else:
    import json

    for name in ["finetuned_r8", "finetuned", "finetuned_attn"]:
        path = Path("reports") / f"{name}.json"
        if path.exists():
            r = json.loads(path.read_text())
            cfg = r.get("training", {}).get("lora", {})
            modules = "all-linear" if len(cfg.get("target_modules", [])) > 4 else "attention"
            print(
                f"r={cfg.get('r', '?'):<3} {modules:<11} "
                f"target {r['fields']['target']['accuracy']:.1%}  "
                f"equipment {r['fields']['equipment']['accuracy']:.1%}  "
                f"joint {r['joint_accuracy']:.1%}"
            )

## 8 · Honest reading

See [`README.md`](../README.md) for the committed results table and the written
conclusion, and [`docs/DATASET.md`](../docs/DATASET.md) for the dataset's known biases —
including the ~1.7% of validation records whose name/label pair also appears in train
under different wording, which biases these numbers slightly upward.

The test split (122 records) has deliberately not been touched. It is reserved for M2, so
that the rigorous evaluation there is not run on a set already used to make decisions here.